In [1]:
!pip install transformers datasets accelerate -q


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset

C:\Users\dhruv\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: False


In [5]:
df = pd.read_csv(r"C:\Users\dhruv\OneDrive\Desktop\Project Ticket\Data\cleaned_data.csv")

df.head()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,...,tag_6,tag_7,tag_8,text,text_clean,char_length,word_count,avg_word_length,unique_word_ratio,type_encoded
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,...,NaN,NaN,NaN,Wesentlicher Sicherheitsvorfall Sehr geehrtes ...,wesentlicher sicherheitsvorfall sehr geehrtes ...,783,84,8.333333,0.869048,1
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,...,NaN,NaN,NaN,"Account Disruption Dear Customer Support Team,...",account disruption dear customer support teamn...,563,84,5.714286,0.833333,1
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,...,NaN,NaN,NaN,Query About Smart Home System Integration Feat...,query about smart home system integration feat...,585,83,6.060241,0.819277,3
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,...,NaN,NaN,NaN,Inquiry Regarding Invoice Details Dear Custome...,inquiry regarding invoice details dear custome...,639,95,5.736842,0.768421,3
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,...,NaN,NaN,NaN,Question About Marketing Agency Software Compa...,question about marketing agency software compa...,732,103,6.116505,0.776699,2


In [6]:
df.shape

(28587, 23)

In [7]:
X = df["text_clean"]
y = df["type_encoded"]

In [8]:
df["type_encoded"].isnull().sum()

np.int64(0)

In [9]:
df_filtered = df.dropna(subset=['text_clean', 'type_encoded'])
X_cleaned = df_filtered["text_clean"]
y_cleaned = df_filtered["type_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned,
    y_cleaned,
    test_size=0.2,
    random_state=42,
    stratify=y_cleaned
)

In [10]:
train_df = pd.DataFrame({
    "text": X_train,
    "label": y_train.astype(int)
})

test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test.astype(int)
})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [11]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [12]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   4%|▍         | 1000/22869 [00:00<00:08, 2504.46 examples/s]

Map: 100%|██████████| 5718/5718 [00:00<00:00, 11811.91 examples/s]


In [13]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

Map: 100%|██████████| 5718/5718 [00:00<00:00, 13382.72 examples/s]


In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
    problem_type="single_label_classification"
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1110.54it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")

    return {
        "accuracy": acc,
        "f1_macro": f1
    }

In [16]:
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",
    save_strategy="no",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=0.1,

    weight_decay=0.01,

    logging_dir="./logs",
    logging_steps=50,

    fp16=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

C:\Users\dhruv\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
trainer.evaluate()

In [ ]:
predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=1)

print("Accuracy:", accuracy_score(y_test, preds))
print("Macro F1:", f1_score(y_test, preds, average="macro"))

print("\nClassification Report:\n")
print(classification_report(y_test, preds))

Accuracy: 0.7997551591465547
Macro F1: 0.7780370068741082

Classification Report:

              precision    recall  f1-score   support

           0       0.92      0.95      0.94       584
           1       0.71      0.90      0.79      2293
           2       0.61      0.30      0.40      1203
           3       0.99      0.98      0.98      1638

    accuracy                           0.80      5718
   macro avg       0.81      0.78      0.78      5718
weighted avg       0.79      0.80      0.78      5718



In [ ]:
trainer.save_model("/content/distilbert_ticket_classifier")
tokenizer.save_pretrained("/content/distilbert_ticket_classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/distilbert_ticket_classifier/tokenizer_config.json',
 '/content/distilbert_ticket_classifier/tokenizer.json')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report:\n")
print(classification_report(y_test, preds))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, preds))

Classification Report:



NameError: name 'preds' is not defined

In [ ]:
import pandas as pd

test_df = pd.DataFrame({
    "text": X_test,
    "actual": y_test,
    "predicted": preds
})

wrong = test_df[test_df["actual"] != test_df["predicted"]]

print("Some wrong predictions:\n")
wrong.head(10)

In [ ]:
label_map = dict(enumerate(df_filtered["type"].astype("category").cat.categories))

print("Label Mapping:")
print(label_map)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_ticket(text, return_probabilities=False):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=1)[0] # Get probabilities for the first (and only) item in batch
    pred_idx = torch.argmax(probabilities).item()
    predicted_class = label_map[pred_idx]

    if return_probabilities:
        all_probabilities = {label_map[i]: prob.item() for i, prob in enumerate(probabilities)}
        return {
            "predicted_class": predicted_class,
            "confidence": probabilities[pred_idx].item(),
            "all_probabilities": all_probabilities
        }
    else:
        return predicted_class

In [ ]:
print(predict_ticket("My payment failed but money got deducted"))

In [ ]:
baseline_accuracy = 0.85
bert_accuracy = accuracy_score(y_test, preds)

print("Baseline Accuracy:", baseline_accuracy)
print("DistilBERT Accuracy:", bert_accuracy)

if bert_accuracy > baseline_accuracy:
    print("DistilBERT performs better")
else:
    print("Baseline model performs better")

In [ ]:
print("FINAL SUMMARY:\n")

print("1. Built baseline model using TF-IDF")
print("2. Built advanced model using DistilBERT")
print("3. Evaluated using accuracy, precision, recall, F1-score")
print("4. Compared both models")

print("\nConclusion:")
print("DistilBERT captures context better but is computationally expensive.")
print("TF-IDF is faster but less powerful.")

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt
from tabulate import tabulate

print("="*80)
print("📊 PER-CLASS PERFORMANCE ANALYSIS")
print("="*80)

# Ensure y_pred is defined as preds
y_pred = preds

# Get numerical unique labels from y_test for sklearn function
numeric_labels = sorted(y_test.unique())

# Get string labels for display from label_map
display_labels = [label_map[label] for label in numeric_labels]

# Get per-class metrics
precision_per_class, recall_per_class, f1_per_class, support = \
    precision_recall_fscore_support(y_test, y_pred, labels=numeric_labels, zero_division=0)

# Create detailed per-class DataFrame
per_class_metrics = pd.DataFrame({
    'Class': display_labels,
    'Precision': precision_per_class,
    'Recall': recall_per_class,
    'F1-Score': f1_per_class,
    'Support': support,
    'Accuracy_Rate': recall_per_class,
    'Error_Rate': 1 - recall_per_class
})

print("\n" + tabulate(per_class_metrics, headers='keys', tablefmt='grid', showindex=False))

# Save per-class metrics
per_class_metrics.to_csv('distilbert_per_class_performance.csv', index=False)
print("\n💾 Saved: distilbert_per_class_performance.csv")

# Visualize per-class performance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Precision, Recall, F1
x = np.arange(len(display_labels))
width = 0.25

axes[0].bar(x - width, precision_per_class, width, label='Precision', color='skyblue')
axes[0].bar(x, recall_per_class, width, label='Recall', color='coral')
axes[0].bar(x + width, f1_per_class, width, label='F1-Score', color='lightgreen')

axes[0].set_xlabel('Ticket Type', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Score', fontsize=12, fontweight='bold')
axes[0].set_title('DistilBERT: Per-Class Performance', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(display_labels, rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0, 1.1])

# Plot 2: Support distribution
axes[1].bar(display_labels, support, color='plum', edgecolor='black')
axes[1].set_xlabel('Ticket Type', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Samples', fontsize=12, fontweight='bold')
axes[1].set_title('Test Set Distribution', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

for i, v in enumerate(support):
    axes[1].text(i, v + max(support)*0.02, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('distilbert_per_class_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💾 Saved: distilbert_per_class_metrics.png")

In [ ]:
print("="*80)
print("🧪 TESTING INFERENCE FUNCTION")
print("="*80)

# Test tickets
test_tickets = [
    "Cannot login to my account. Getting error 404. Please help urgently!",
    "I need a refund for my last payment. I was charged twice.",
    "How do I reset my password? I forgot it.",
    "The entire system is down. Production environment not responding.",
    "I want to change my email address in account settings.",
    "My payment was declined but money was deducted from my account.",
    "Need help configuring the API integration with Salesforce.",
    "When will the new feature be released? Any timeline?"
]

print("\n📋 Testing Predictions:\n")

for i, ticket in enumerate(test_tickets, 1):
    print(f"{'─'*80}")
    print(f"Test Case {i}:")
    print(f"{'─'*80}")
    print(f"Ticket: {ticket}")

    # Correct function name and pass return_probabilities=True
    result = predict_ticket(ticket, return_probabilities=True)

    print(f"\n✅ Predicted Class: {result['predicted_class']}")
    print(f"📊 Confidence: {result['confidence']:.2%}")

    print(f"\n📊 All Class Probabilities:")
    for class_name, prob in sorted(result['all_probabilities'].items(),
                                  key=lambda x: x[1], reverse=True):
        bar_length = int(prob * 50)
        bar = '█' * bar_length
        print(f"   {class_name:15s} {bar} {prob:.2%}")

    print()

print("✅ All test cases completed!")